# N146 · FFT、NTT与卷积拓展

**目标：** 理解卷积和系数乘积并区分浮点与模域结果。

**先修：** N063, N128, N130。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 多项式卷积；DFT；分治FFT；补零；舍入误差；NTT模数与单位根。

## 从问题到算法

卷积系数c[k]=∑a[i]b[k−i]是多项式乘法。DFT把卷积变成逐点乘法；按偶数、奇数下标拆分可递归复用较短DFT，得到蝶形网络。必须补零到至少len(a)+len(b)−1，否则得到循环卷积而发生折叠。FFT用复数浮点，NTT在满足单位根条件的有限域中计算，两者的精确性范围不同。

## 最小实现

**本章接口：** `convolve_naive(a, b)`、`fft(values, invert=False)`、`convolve_fft(a, b)`、`ntt(values, invert, mod, primitive_root)`

In [1]:
from cmath import exp,pi
from functools import cache
from math import isqrt

def convolve_naive(a,b):
    if not a or not b: return []
    out=[0]*(len(a)+len(b)-1)
    for i,x in enumerate(a):
        for j,y in enumerate(b): out[i+j]+=x*y
    return out

def fft(values,invert=False):
    a=list(map(complex,values)); n=len(a)
    if not n: return []
    if n&(n-1): raise ValueError('power-of-two transform length required')
    j=0
    for i in range(1,n):
        bit=n>>1
        while j&bit: j^=bit; bit>>=1
        j^=bit
        if i<j: a[i],a[j]=a[j],a[i]
    length=2
    while length<=n:
        root=exp((2j if invert else -2j)*pi/length)
        for start in range(0,n,length):
            w=1
            for offset in range(length//2):
                u=a[start+offset]; v=a[start+offset+length//2]*w
                a[start+offset]=u+v; a[start+offset+length//2]=u-v; w*=root
        length*=2
    return [x/n for x in a] if invert else a

def convolve_fft(a,b):
    # Returns floating approximations, not an exact arbitrary-integer convolution.
    if not a or not b: return []
    size=1; result_length=len(a)+len(b)-1
    while size<result_length: size*=2
    left=fft(list(a)+[0]*(size-len(a))); right=fft(list(b)+[0]*(size-len(b)))
    return [x.real for x in fft([x*y for x,y in zip(left,right)],True)[:result_length]]

@cache
def _is_prime(mod):
    if mod<2: return False
    if mod%2==0: return mod==2
    return all(mod%d for d in range(3,isqrt(mod)+1,2))

def ntt(values,invert,mod,primitive_root):
    a=[x%mod for x in values]; n=len(a)
    if not n: return []
    if n&(n-1) or not _is_prime(mod) or (mod-1)%n: raise ValueError('prime modulus and power-of-two length dividing mod-1 required')
    root_n=pow(primitive_root,(mod-1)//n,mod)
    if pow(root_n,n,mod)!=1 or n>1 and pow(root_n,n//2,mod)==1: raise ValueError('root does not provide required order')
    j=0
    for i in range(1,n):
        bit=n>>1
        while j&bit: j^=bit; bit>>=1
        j^=bit
        if i<j: a[i],a[j]=a[j],a[i]
    length=2
    while length<=n:
        root=pow(primitive_root,(mod-1)//length,mod)
        if invert: root=pow(root,mod-2,mod)
        for start in range(0,n,length):
            w=1
            for offset in range(length//2):
                u=a[start+offset]; v=a[start+offset+length//2]*w%mod
                a[start+offset]=(u+v)%mod; a[start+offset+length//2]=(u-v)%mod; w=w*root%mod
        length*=2
    if invert:
        inv=pow(n,mod-2,mod); a=[x*inv%mod for x in a]
    return a

def convolve_ntt(a,b,mod=998244353,primitive_root=3):
    if not a or not b: return []
    size=1; length=len(a)+len(b)-1
    while size<length: size*=2
    left=ntt(list(a)+[0]*(size-len(a)),False,mod,primitive_root)
    right=ntt(list(b)+[0]*(size-len(b)),False,mod,primitive_root)
    return ntt([x*y%mod for x,y in zip(left,right)],True,mod,primitive_root)[:length]

## 正确性、前提与复杂度

DFT在单位根上求值，多项式乘法在每个点变成乘法，逆变换恢复系数。蝶形利用ω_N^(2k)=ω_(N/2)^k和ω_N^(k+N/2)=−ω_N^k。NTT要求模数为素数、变换长度整除p−1且所用根确有该阶，才能使用逆长度与逆单位根。当前代码明确返回FFT近似值，不把舍入接近整数当成任意大整数正确性的证明。

**代价：** 直接卷积O(nm)；补零长度N的FFT/NTT O(N log N)算术操作O(N)空间。NTT参数初次试除验证素性另有O(√p)代价并缓存。FFT测试仅覆盖长度<30、系数绝对值≤100且明确容差；更大范围需误差分析。NTT只保证模p结果，恢复更大有符号整数需额外模数与系数界。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

a,b=[1,2],[3,4]; exact=convolve_naive(a,b); approximate=convolve_fft(a,b); modular=convolve_ntt(a,b)
show_table(['系数下标','直接整数卷积','浮点FFT近似','NTT模结果'],[(i,exact[i],approximate[i],modular[i]) for i in range(len(exact))])
values=[1,2,3,4]; spectrum=fft(values)
show_table(['频率','DFT实部','DFT虚部'],[(k,x.real,x.imag) for k,x in enumerate(spectrum)])

系数下标,直接整数卷积,浮点FFT近似,NTT模结果
0,3,3.0,3
1,10,10.0,10
2,8,8.0,8


频率,DFT实部,DFT虚部
0,10.0,0.0
1,-2.0,2.0
2,-2.0,0.0
3,-1.9999999999999998,-2.0


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
from random import Random
from math import isclose
rng=Random(146)
assert convolve_naive([1,2],[3,4])==[3,10,8]
assert convolve_ntt([1,2],[3,4])==[3,10,8]
for n in [1,2,4,8,16,32]:
    values=[rng.randrange(-100,101) for _ in range(n)]
    restored=fft(fft(values),True)
    assert all(abs(a-b)<1e-9 for a,b in zip(values,restored))
    assert ntt(ntt(values,False,998244353,3),True,998244353,3)==[x%998244353 for x in values]
for _ in range(140):
    a=[rng.randrange(-100,101) for _ in range(rng.randrange(1,30))]; b=[rng.randrange(-100,101) for _ in range(rng.randrange(1,30))]
    expected=convolve_naive(a,b); approximate=convolve_fft(a,b)
    assert all(isclose(x,y,rel_tol=1e-10,abs_tol=1e-7) for x,y in zip(expected,approximate))
    assert convolve_ntt(a,b)==[x%998244353 for x in expected]
assert ntt([1,2,3,4],False,17,3)
try: ntt([1,2,3,4],False,21,2)
except ValueError: pass
else: raise AssertionError('composite modulus must be rejected')
assert convolve_fft([],[])==convolve_ntt([],[])==[]
print('N146: 所有本章断言通过')

N146: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 用小整数多项式对拍。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 说明LC43先掌握竖式乘法，不把FFT当必需解法。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 43. Multiply Strings（extension）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。